In [17]:
import os
import sys
from pathlib import Path

root_dir = str(Path(os.getcwd()).parent)

if root_dir not in sys.path:
    sys.path.append(root_dir)

output_dir = "Outputs"
os.makedirs(output_dir, exist_ok=True)

<div style="background-color: #51b1da; color: white; padding: 30px; border-radius: 0px;">
<h1 style="margin: 0;  color: #04335a">The Wave Equation </h3>
</div>

Imports:

In [18]:
import numpy as np
from IPython.display import Video, HTML

from Solver_Classes.solwe import Wave_Equation

---

## Dynamic Propagation

The 2D Wave Equation is a hyperbolic partial differential equation that models the propagation of oscillations—like ripples on a water surface or acoustic waves in a membrane. It is defined as:

$$\frac{\partial^2 u}{\partial t^2} = c^2 \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right)$$

To solve this numerically, our `Wave_Equation` class utilizes a **Leap-Frog integration scheme**, evaluating both the current and previous time steps to compute spatial acceleration.

**Geometric Masking:** Unlike standard rectangular domains, this solver enforces **zero-boundary Dirichlet conditions** across irregular spaces. By applying a mathematical mask at every time step, we can perfectly simulate wave reflections inside custom boundaries (like an elliptical pool).

---

### Defining the Domain & Conditions:

In [19]:
#                  1. System Parameters
N = 100            # Grid resolution (N x N)
H = 2.0            # Size of the spatial domain
time_s = 1000      # Number of time steps to simulate

# 2. Geometric Mask 
#    (Defining an Elliptical Boundary)

def boundary_mask(x, y):
    x_c, y_c, R = H/2, H/2, H/2
    return np.where(0.5*(x - x_c)**2 + (y - y_c)**2 <= R**2 / 2, 1, 0)

# 3. Initial Condition 
#    (Two Gaussian wave drops inside the ellipse)

def initial_condition(x, y):
    a = 1.73205080757
    func = 2*np.exp(-50*((x - a)**2 + (y - H/2)**2)) + 2*np.exp(-50*((x - (2 - a))**2 + (y - H/2)**2))
    
    mask_grid = boundary_mask(x, y)
    return np.where(mask_grid == 1, func, 0)

### Running the Simulation:

In [20]:
# Paths

video_2d_target = os.path.join(output_dir, "Wave_Equation_Simulation_2D.mp4")
video_3d_target = os.path.join(output_dir, "Wave_Equation_Simulation_3D.mp4")

In [21]:
WE = Wave_Equation(
    H=H, 
    n=N, 
    simulation_time=time_s, 
    IC=initial_condition, 
    mask=boundary_mask, 
    number_of_workers=15 # !!! Depends on your computer !!!
)

print("Solving the 2D Wave Equation...")
sol, vmax, vmin = WE.solve_Wave_Equation_mask_DBC()

raw_limit = max(abs(vmax), abs(vmin))
brightness_factor = 2.5  
v_limit = raw_limit / brightness_factor

Solving the 2D Wave Equation...


100%|██████████| 998/998 [00:00<00:00, 5094.91it/s]


Total execution time: 0h 0m 0s


Saving the Animations:

In [22]:
print("\nInitiating 2D Rendering Pipeline...")
WE.render_Wave_Eqn_2D(sol, v_limit, -v_limit, video_filename=video_2d_target)

print("\nInitiating 3D Rendering Pipeline...")
WE.render_Wave_Eqn_3D(sol, v_limit, -v_limit, video_filename=video_3d_target)


Initiating 2D Rendering Pipeline...
Rendering frames in parallel...


Rendering: 100%|██████████| 1000/1000 [00:24<00:00, 40.98it/s]


Sample frame resolution: 769x770 → using 770x770 for video
Stitching frames into video...
Deleting frames folder...
Video saved to Outputs/Wave_Equation_Simulation_2D.mp4
Total execution time: 0h 0m 32s

Initiating 3D Rendering Pipeline...
Rendering frames in parallel...


Rendering: 100%|██████████| 1000/1000 [01:07<00:00, 14.77it/s]


Sample frame resolution: 769x770 → using 770x770 for video
Stitching frames into video...
Deleting frames folder...
Video saved to Outputs/Wave_Equation_Simulation_3D.mp4
Total execution time: 0h 1m 23s


---

### Inspecting the Results:
The two initial Gaussian disturbances propagate, hit the elliptical Dirichlet boundaries, and reflect back onto themselves.

In [23]:
Video("Outputs/Wave_Equation_Simulation_2D.mp4", width=700, html_attributes="controls autoplay loop muted")

In [24]:
Video("Outputs/Wave_Equation_Simulation_3D.mp4", width=700, html_attributes="controls autoplay loop muted")